## 11. Optional: Hyperparameter Tuning

Tuning tries multiple parameter combinations with `TimeSeriesSplit`. This can take a long time. Use it only after the simple training run works.

When `PERSIST_TUNED_MODELS = True`, each tuned candidate is refit on the full training split and saved immediately after that refit completes under `runs/<timestamp>_<model>/tuned_modelN/`. The root-level `model.joblib` remains the best model for compatibility with the evaluation notebook.


In [1]:
from pathlib import Path
import os
os.environ["OMP_NUM_THREADS"] = "48"
import sys
import importlib
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# Make local imports work when the notebook is opened from another folder.
PIPELINE_DIR = Path.cwd()
if not (PIPELINE_DIR / 'loader.py').exists():
    PIPELINE_DIR = Path.cwd() / 'pipeline'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.append(str(PIPELINE_DIR))

from loader import LoaderStorage
from targets import add_delay_class_target, DELAY_CLASS_ORDER
from split import chronological_train_val_test_split
from features import  check_columns
from evaluate import plotConfusionMatrix
import models
importlib.reload(models)
import train
importlib.reload(train)
from train import TrainingConfig, run_training


In [2]:
# Change these paths to your actual dataset location.
DATA_ROOT = "s3://data-mining"
INPUT_PATH = 'data/features/n3_feature_engineered.parquet'

# Column names used by the current pipeline.
DELAY_COLUMN = 'ArrDelayMinutes'
TIME_COLUMN = 'CRSDepDateTime_UTC'
TARGET_COLUMN = 'delay_class'

# Use a small sample while learning/debugging. Set to 1.0 for the final run.
SAMPLE_FRAC = 1

# Good first choices: 'dummy', 'logistic_regression', 'random_forest', 'hist_gradient_boosting', 'xgboost', 'svc'.
MODEL_NAMES = ['logistic_regression_pipeline', 'xgboost',  'random_forest', 'hist_gradient_boosting'] #'svc']
N_ITER = 5
CV_SPLITS = 3
PERSIST_TUNED_MODELS = True

OUTPUT_DIR = 'outputs/training_notebook_tuned'


In [ ]:
%%time
%time
# Uncomment this cell when you are ready for a slower tuning run.
for n in MODEL_NAMES:
  print(f"TRAINING {n}")
  tuned_config = TrainingConfig(
      data_root=DATA_ROOT,
      input_path=INPUT_PATH,
      output_dir=OUTPUT_DIR,
      model_name=n,
      delay_column=DELAY_COLUMN,
      target_column=TARGET_COLUMN,
      time_column=TIME_COLUMN,
      sample_frac=SAMPLE_FRAC,
      weights=None, # ignored during tuning; class weights are sampled from intervals in RandomizedSearchCV.
      tune=True,
      n_iter=N_ITER,
      cv_splits=CV_SPLITS,
      persist_tuned_models=PERSIST_TUNED_MODELS,
  )
  tuned_metrics,best_params = run_training(tuned_config)
  display(pd.Series(tuned_metrics, name='tuned_pipeline_metrics'))
  %time
  if best_params:
      print("Best hyperparameters found during tuning:")
      for param, value in best_params.items():
          print(f"{param}: {value}")

CPU times: user 4 μs, sys: 1e+03 ns, total: 5 μs
Wall time: 8.11 μs
TRAINING logistic_regression_pipeline
Fitting 3 folds for each of 5 candidates, totalling 15 fits


/home/justus/ie500_data_mining_project/pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/justus/ie500_data_mining_project/pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as sho